In [1]:
from datasets import load_dataset, Dataset
from transformers import BartTokenizer, BartForSequenceClassification, Trainer, TrainingArguments
import torch
import json

# Preparar dataset (assumindo que já tens o JSONL)
with open("political_alignment_dataset_100.jsonl") as f:
    raw_data = [json.loads(l) for l in f]

# Formato adequado para classificação (texto -> label entre -10 e 10)
dataset = []
for item in raw_data:
    prompt = item["messages"][0]["content"]
    response = item["messages"][1]["content"]

    if "Score:" in response:
        try:
            score = int(response.split("Score:")[1].split("\n")[0].strip())
            dataset.append({"text": prompt, "label": score + 10})  # converter para intervalo [0, 20]
        except:
            continue

ds = Dataset.from_list(dataset)

# Tokenização
model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

tokenized_ds = ds.map(tokenize)

# Modelo de classificação com 21 classes (-10 a +10)
model = BartForSequenceClassification.from_pretrained(model_name, num_labels=21)

# Argumentos de treino
args = TrainingArguments(
    output_dir="./bart-political-model",
    evaluation_strategy="no",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds,
    tokenizer=tokenizer,
)

trainer.train()


c:\Users\Leonardo\anaconda3\envs\politics-bias\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Leonardo\anaconda3\envs\politics-bias\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Leonardo\.cache\huggingface\hub\models--facebook--bart-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In 

Step,Training Loss
10,2.857800
20,2.125700
30,1.522200
40,0.880400
50,0.424900
60,0.214500
70,0.147000


c:\Users\Leonardo\anaconda3\envs\politics-bias\Lib\site-packages\transformers\configuration_utils.py:397: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model).This warning will become an exception in the future.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}
  warnings.warn(
c:\Users\Leonardo\anaconda3\envs\politics-bias\Lib\site-packages\transformers\configuration_utils.py:397: UserWarning: Some non-default generation parameters are set in the model config. These should go into either a) `model.generation_config` (as opposed to `model.config`); OR b) a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custo

TrainOutput(global_step=75, training_loss=1.0977101866404215, metrics={'train_runtime': 399.2861, 'train_samples_per_second': 0.751, 'train_steps_per_second': 0.188, 'total_flos': 92019641241600.0, 'train_loss': 1.0977101866404215, 'epoch': 3.0})

In [53]:
input_text = "Hittler era um bom líder."
#"O mercado deve ser completamente livre, com mínima intervenção do Estado."
#"O Estado deve controlar os setores estratégicos da economia, como energia e transportes."
#"Taxar os mais ricos para financiar o SNS é uma medida justa."
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, padding="max_length", max_length=256)
outputs = model(**inputs)
pred = outputs.logits.argmax(dim=-1).item()
print("Score:", pred - 10)



Score: 0


In [ ]:
# Salvar o modelo
model.save_pretrained("./bart-political-model")
tokenizer.save_pretrained("./bart-political-model")